# Liar Unified Schema Process

This is where all the steps are taken to convert the Liar dataset to the agreed unified schema.

Unified Schema:
- `dataset`, `id`, `split`
- `label_raw`, `label` (true/mixed/false), `label_confidence` ∈ {gold, weak}, `label_raw_source`
- **Short text**: `claim_text` (tweet, post, or title)
- **Long text (optional)**: `article_text` (parsed), `content_status` ∈ {full_article, partial, title_only}
- URLs/meta: `post_url`/`news_url`, `archive_url`, `is_archived`, `source_domain`

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

column_names = ['claim_id','label','statement','subject','speaker',
                'job_title','state_info','party_affiliation','barely_true_counts','false_counts','half_true_counts','mostly_true_counts','pants_fire_counts','context']

train = pd.read_csv('../../raw/liar/train.tsv',sep='\t',header=None,names=column_names)
val = pd.read_csv('../../raw/liar/valid.tsv',sep='\t',header=None,names=column_names)
test = pd.read_csv('../../raw/liar/test.tsv',sep='\t',header=None,names=column_names)

In [2]:
train.head()

,claim_id,label,statement,subject,speaker,job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


In [3]:
val.head()

,claim_id,label,statement,subject,speaker,job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_fire_counts,context
0,12134.json,barely-true,We have less Americans working now than in the...,"economy,jobs",vicky-hartzler,U.S. Representative,Missouri,republican,1,0,1,0,0,an interview with ABC17 News
1,238.json,pants-fire,"When Obama was sworn into office, he DID NOT u...","obama-birth-certificate,religion",chain-email,NaN,NaN,none,11,43,8,5,105,NaN
2,7891.json,false,Says Having organizations parading as being so...,"campaign-finance,congress,taxes",earl-blumenauer,U.S. representative,Oregon,democrat,0,1,1,1,0,a U.S. Ways and Means hearing
3,8169.json,half-true,Says nearly half of Oregons children are poor.,poverty,jim-francesconi,Member of the State Board of Higher Education,Oregon,none,0,1,1,1,0,an opinion article
4,929.json,half-true,On attacks by Republicans that various program...,"economy,stimulus",barack-obama,President,Illinois,democrat,70,71,160,163,9,interview with CBS News


In [4]:
test.head()

,claim_id,label,statement,subject,speaker,job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_fire_counts,context
0,11972.json,true,Building a wall on the U.S.-Mexico border will...,immigration,rick-perry,Governor,Texas,republican,30,30,42,23,18,Radio interview
1,11685.json,false,Wisconsin is on pace to double the number of l...,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,0,0,0,a news conference
2,11096.json,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,51,37,61,comments on ABC's This Week.
3,5209.json,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,3,1,1,a radio show
4,9524.json,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,2,2,7,a web video


## Statistical tests for importance of Party affiliation, Speaker, and Job title columns

In [5]:
min_count = 30
top = train['party_affiliation'].value_counts().loc[lambda s: s >= min_count].index
train['party2'] = train['party_affiliation'].where(train['party_affiliation'].isin(top), 'Other')

ct = pd.crosstab(train['party2'], train['label'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
n = ct.values.sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r,k) - 1)))

print(ct)
print(f"chi2={chi2:.2f}, p={p:.3g}, Cramér's V={cramers_v:.3f}")

label         barely-true  false  half-true  mostly-true  pants-fire  true
party2                                                                    
Other                   6     20         20           17           9    17
activist                7      7          9           11           0     5
columnist               5     10          3            5           1    11
democrat              463    511        750          801         153   658
independent            19     18         27           51           3    29
journalist              5     10          6           11           1     5
libertarian             4      6         12            9           3     6
newsmaker               7      9         11           11           3    15
none                  261    326        327          315         269   246
organization           45     50         59           24          17    24
republican            832   1028        890          707         380   660
chi2=452.00, p=4.02e-66, 

Statistically with such a high chi-square deviation value of observed from the expected values for each party affiliation and counts and a low p-value, this rejects independence very strongly. The test says “party and label are not independent” (very strong statistical significance), but the strength of that relationship is small (Cramér’s V ≈ 0.094). That pattern usually happens when you have a large sample: tiny differences across many rows become statistically significant, but are not practically large.

This means that we can drop the party affiliation column as it might not add much predictive power to the model.

In [6]:
min_count = 30
top = train['speaker'].value_counts().loc[lambda s: s >= min_count].index
train['speaker2'] = train['speaker'].where(train['speaker'].isin(top), 'Other')

ct = pd.crosstab(train['speaker'], train['label'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
n = ct.values.sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r,k) - 1)))

print(ct)
print(f"chi2={chi2:.2f}, p={p:.3g}, Cramér's V={cramers_v:.3f}")

label                            barely-true  false  half-true  mostly-true  \
speaker                                                                       
18-percent-american-public                 0      0          0            0   
60-plus-association                        1      0          0            0   
AARP                                       0      0          0            1   
Arizona-Citizens-Defense-League            0      1          1            0   
Ballesteros                                1      0          0            0   
...                                      ...    ...        ...          ...   
yvette-mcgee-brown                         0      0          0            0   
zack-space                                 0      0          0            0   
zell-miller                                0      1          0            2   
zephyr-teachout                            0      0          1            0   
zoe-lofgren                                0      0 

Same thing for the speaker column.

In [7]:
min_count = 20
top = train['job_title'].value_counts().loc[lambda s: s >= min_count].index
train['job2'] = train['job_title'].where(train['job_title'].isin(top), 'Other')

ct = pd.crosstab(train['job2'], train['label'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
n = ct.values.sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r,k) - 1)))

print(ct)
print(f"chi2={chi2:.2f}, p={p:.3g}, Cramér's V={cramers_v:.3f}")

label                                     barely-true  false  half-true  \
job2                                                                      
Attorney                                           13     21         22   
Attorney General                                    2     10          9   
Businessman                                         8      7          4   
Candidate for U.S. Senate and physician             5      4          8   
Co-host on CNN's "Crossfire"                       14     15         20   
Columnist                                           4      6          1   
Congressman                                        10     24         17   
Congresswoman                                       8     18          6   
Former governor                                    27     28         47   
Governor                                           65     75         93   
Governor of New Jersey                              8     17         16   
Governor of Ohio as of Ja

Same with the job title.

## Total vote counts distributions

In [8]:
train['total_counts'] = train['barely_true_counts']+train['half_true_counts']+train['mostly_true_counts'] + train['false_counts']+ train['pants_fire_counts']

In [9]:
val['total_counts'] = val['barely_true_counts']+val['half_true_counts']+val['mostly_true_counts'] + val['false_counts']+ val['pants_fire_counts']

In [10]:
test['total_counts'] = test['barely_true_counts']+test['half_true_counts']+test['mostly_true_counts'] + test['false_counts']+ test['pants_fire_counts']

In [11]:
bins = [0,10,50,100,300,500,np.inf]
labels = ['<10','10-49','50-99','100-299','300-499','500+']
train['count_category'] = pd.cut(train['total_counts'],bins=bins,labels=labels,right=False,include_lowest=False)
train['count_category'].value_counts(normalize=True)

count_category
<10        0.477632
10-49      0.244286
100-299    0.129029
50-99      0.074722
300-499    0.074331
500+       0.000000
Name: proportion, dtype: float64

In [12]:
val['count_category'] = pd.cut(val['total_counts'],bins=bins,labels=labels,right=False,include_lowest=False)
val['count_category'].value_counts(normalize=True)

count_category
<10        0.455607
10-49      0.255452
100-299    0.132399
50-99      0.080218
300-499    0.076324
500+       0.000000
Name: proportion, dtype: float64

In [13]:
test['count_category'] = pd.cut(test['total_counts'],bins=bins,labels=labels,right=False,include_lowest=False)
test['count_category'].value_counts(normalize=True)

count_category
<10        0.441989
10-49      0.269140
100-299    0.127861
50-99      0.086030
300-499    0.074980
500+       0.000000
Name: proportion, dtype: float64

In [14]:
ct = pd.crosstab(train['count_category'], train['label'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
n = ct.values.sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r,k) - 1)))

print(ct)
print(f"chi2={chi2:.2f}, p={p:.3g}, Cramér's V={cramers_v:.3f}")

label           barely-true  false  half-true  mostly-true  pants-fire  true
count_category                                                              
<10                     766    949        998          955         345   877
10-49                   446    483        526          477         187   382
50-99                   129    142        160          129          97   108
100-299                 209    251        267          244         150   200
300-499                 104    168        163          157          60   109
chi2=74.81, p=2.93e-08, Cramér's V=0.043


We can see that the same occurs with the total count categories columns. Meaning we can drop them as well.

## Unifiying schema

In [15]:
train.columns

Index(['claim_id', 'label', 'statement', 'subject', 'speaker', 'job_title',
       'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts',
       'half_true_counts', 'mostly_true_counts', 'pants_fire_counts',
       'context', 'party2', 'speaker2', 'job2', 'total_counts',
       'count_category'],
      dtype='object')

In [16]:
val.columns

Index(['claim_id', 'label', 'statement', 'subject', 'speaker', 'job_title',
       'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts',
       'half_true_counts', 'mostly_true_counts', 'pants_fire_counts',
       'context', 'total_counts', 'count_category'],
      dtype='object')

In [17]:
test.columns

Index(['claim_id', 'label', 'statement', 'subject', 'speaker', 'job_title',
       'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts',
       'half_true_counts', 'mostly_true_counts', 'pants_fire_counts',
       'context', 'total_counts', 'count_category'],
      dtype='object')

In [18]:
train.drop(columns=['party2','job2','speaker2'],inplace=True)

In [19]:
def unified_schema(df:pd.DataFrame, dataset:str = "liar", split:str = 'Train') -> pd.DataFrame:
    temp = df.copy()
    temp['dataset'], temp['split'] = dataset,split
    temp['raw_label'] = temp['label']
    temp['label']=temp['label'].map({'true': 'true','mostly-true':'mixed','half-true':'mixed','barely-true':'mixed','false':'false','pants-fire':'false'})
    temp.rename(columns={'statement':'claim_text'},inplace=True)
    temp['content_status'] = 'title_only'
    temp['label_confidence'] = np.where(temp['count_category'] == '<10', 'weak', 'gold')

    temp.drop(['state_info','barely_true_counts','false_counts','half_true_counts','mostly_true_counts','pants_fire_counts','speaker','party_affiliation','job_title','total_counts','count_category'],inplace=True,axis=1)
    return temp

In [20]:
uni_train = unified_schema(train,'liar','train')
uni_train

,claim_id,label,claim_text,subject,context,dataset,split,raw_label,content_status,label_confidence
0,2635.json,false,Says the Annies List political group supports ...,abortion,a mailer,liar,train,false,title_only,weak
1,10540.json,mixed,When did the decline of coal start? It started...,"energy,history,job-accomplishments",a floor speech.,liar,train,half-true,title_only,weak
2,324.json,mixed,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,Denver,liar,train,mostly-true,title_only,gold
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,a news release,liar,train,false,title_only,gold
4,9028.json,mixed,The economic turnaround started at the end of ...,"economy,jobs",an interview on CNN,liar,train,half-true,title_only,gold
...,...,...,...,...,...,...,...,...,...,...
10235,5473.json,mixed,There are a larger number of shark attacks in ...,"animals,elections","interview on ""The Colbert Report""",liar,train,mostly-true,title_only,weak
10236,3408.json,mixed,Democrats have now become the party of the [At...,elections,an interview,liar,train,mostly-true,title_only,weak
10237,3959.json,mixed,Says an alternative to Social Security that op...,"retirement,social-security",a Republican presidential debate,liar,train,half-true,title_only,gold
10238,2253.json,false,On lifting the U.S. Cuban embargo and allowing...,"florida,foreign-policy",a televised debate on Miami's WPLG-10 against ...,liar,train,false,title_only,weak


In [21]:
uni_val = unified_schema(val,'liar','valid')
uni_test = unified_schema(test,'liar','test')

## Save to parquet files

In [22]:
uni_train.to_parquet('unified_liar_train.parquet', index=False)
uni_val.to_parquet('unified_liar_valid.parquet', index=False)
uni_test.to_parquet('unified_liar_test.parquet', index=False)